# Analisis exploratorio de fraude bancario

Este notebook presenta un analisis exploratorio del dataset PaySim con foco en comprender el comportamiento del fraude transaccional.

El objetivo es construir una lectura clara y profesional de los datos, identificando patrones iniciales que permitan diferenciar transacciones fraudulentas de transacciones normales. En esta etapa no se desarrollan modelos predictivos; el foco esta en interpretacion, calidad del analisis y comunicacion de hallazgos.

## 0. Preparacion del entorno

Antes de iniciar el analisis se configuran las librerias, el estilo visual de los graficos y las funciones auxiliares del proyecto.

La instalacion de dependencias se deja como una celda opcional, pensada solo para entornos nuevos o kernels donde falte alguna libreria.

In [ ]:
# Celda opcional: ejecutar solo si falta alguna libreria del proyecto.
# En notebooks, %pip instala paquetes en el kernel activo.
# %pip install -r ../requirements.txt

In [ ]:
# Imports base para analisis exploratorio.
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Configuracion para que el notebook pueda ejecutarse desde la raiz o desde notebooks/.
RUTA_ACTUAL = Path.cwd()
RUTA_PROYECTO = RUTA_ACTUAL.parent if RUTA_ACTUAL.name == "notebooks" else RUTA_ACTUAL
sys.path.append(str(RUTA_PROYECTO))

# Configuracion visual y de salida.
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.2f}".format)
sns.set_theme(style="whitegrid", palette="deep")
PALETA_FRAUDE = {"No fraude": "#4C78A8", "Fraude": "#E45756"}
ETIQUETAS_FRAUDE = {0: "No fraude", 1: "Fraude"}

from src.data.cargar_datos import cargar_csv, listar_archivos_csv, obtener_resumen_carga
from src.analysis.eda_utils import (
    comparar_fraude_vs_no_fraude,
    distribucion_variable,
    resumen_datos,
    resumen_general,
    tasa_fraude_por_grupo,
)
from src.features.generar_variables import crear_variables_temporales
from src.visualization.graficos import configurar_estilo

configurar_estilo()

## 1. Carga unica de datos

El dataset se carga una sola vez desde la carpeta `data/raw/`, evitando duplicar lecturas o generar inconsistencias durante el analisis.

El archivo utilizado corresponde al primer CSV disponible en esa carpeta. Para conservar una lectura completa del dataset se mantiene `n_filas=None`; si el archivo resulta pesado para el equipo, se puede usar una lectura parcial de forma temporal.

In [ ]:
# Carga unica del dataset.
archivos_csv = listar_archivos_csv(carpeta="raw")

if not archivos_csv:
    raise FileNotFoundError(
        "No se encontro ningun CSV en data/raw. "
        "Agrega el archivo de PaySim en esa carpeta antes de continuar."
    )

nombre_archivo = archivos_csv[0].name
datos = cargar_csv(nombre_archivo=nombre_archivo, carpeta="raw", n_filas=None)

# Validacion minima de columnas necesarias para este EDA.
columnas_requeridas = {"step", "type", "amount", "isFraud"}
columnas_faltantes = columnas_requeridas - set(datos.columns)

if columnas_faltantes:
    raise ValueError(f"Faltan columnas requeridas para el EDA: {columnas_faltantes}")

# Normalizamos isFraud para evitar problemas si el CSV la carga como texto.
datos["isFraud"] = pd.to_numeric(datos["isFraud"], errors="coerce").astype("Int64")
datos = datos.assign(clase_fraude=datos["isFraud"].map(ETIQUETAS_FRAUDE))

# Muestra opcional solo para graficos pesados. No reemplaza el dataset original.
MAX_FILAS_GRAFICOS = 100_000
datos_grafico = (
    datos.sample(MAX_FILAS_GRAFICOS, random_state=42)
    if len(datos) > MAX_FILAS_GRAFICOS
    else datos.copy()
)
print(f"Archivo cargado: {nombre_archivo}")
obtener_resumen_carga(datos)

## 2. Comprension de los datos

En esta seccion se revisa la estructura general del dataset: dimensiones, primeras filas, tipos de datos y valores faltantes.

Esta revision permite confirmar que la base fue cargada correctamente y que las variables principales para el analisis de fraude estan disponibles.

In [ ]:
# Primeras filas para entender la forma de los registros.
datos.head()

In [ ]:
# Dimensiones generales del dataset.
print(f"Filas: {datos.shape[0]:,}")
print(f"Columnas: {datos.shape[1]:,}")

In [ ]:
# Informacion de columnas, tipos de datos y valores no nulos.
datos.info()

In [ ]:
# Tabla resumida de tipos, nulos y valores unicos.
resumen_datos(datos)

**Interpretacion:** el dataset presenta una estructura clara donde la variable objetivo `isFraud` permite identificar directamente si una transaccion corresponde a fraude o no.

La mayoria de las variables son numericas, lo cual resulta adecuado para este tipo de analisis, ya que el foco principal esta en el comportamiento de montos y saldos. Las variables categoricas son limitadas y se concentran principalmente en el tipo de transaccion, aportando valor para segmentar el analisis.

## 3. Estadistica descriptiva

Las metricas descriptivas permiten observar la escala, dispersion y posibles valores extremos de las variables numericas.

En datos financieros, estas metricas deben interpretarse con cautela, ya que algunas transacciones de gran magnitud pueden distorsionar la media.

In [ ]:
# Resumen estadistico de variables numericas.
resumen_general(datos)

**Interpretacion:** a partir de las metricas descriptivas se observa que los valores maximos de las variables monetarias son significativamente superiores a sus valores promedio.

Esto evidencia una distribucion altamente asimetrica, con una fuerte concentracion de datos en valores bajos y una cola extendida hacia valores altos. Este comportamiento es esperable en datos financieros, donde pueden existir transacciones de gran magnitud que distorsionan la media.

## 4. Analisis de la variable objetivo

La variable objetivo del analisis es `isFraud`, que indica si una transaccion fue fraudulenta o no.

Antes de explorar patrones, es fundamental cuantificar la proporcion de cada clase, ya que los problemas de fraude suelen presentar un fuerte desbalance entre eventos normales y fraudulentos.

In [ ]:
# Conteo y porcentaje de clases de la variable objetivo.
distribucion_fraude = distribucion_variable(datos, "clase_fraude")
distribucion_fraude

In [ ]:
# Grafico simple de la variable objetivo.
plt.figure(figsize=(7, 4))
ax = sns.countplot(
    data=datos,
    x="clase_fraude",
    hue="clase_fraude",
    order=["No fraude", "Fraude"],
    palette=PALETA_FRAUDE,
    legend=False,
)
ax.set_title("Distribucion de transacciones fraudulentas y no fraudulentas")
ax.set_xlabel("Clase")
ax.set_ylabel("Cantidad de transacciones")
plt.show()

**Interpretacion:** el dataset presenta un fuerte desbalance entre clases, donde las transacciones no fraudulentas representan practicamente la totalidad de los registros.

Este comportamiento es coherente con escenarios reales de fraude financiero, donde los eventos fraudulentos suelen ser poco frecuentes en comparacion con el volumen total de transacciones. Este desbalance debe considerarse al interpretar patrones, ya que una diferencia pequena en porcentaje puede representar un riesgo relevante.

## 5. Distribucion de montos

La variable `amount` representa el monto de la transaccion y es una de las variables mas relevantes para comprender el comportamiento financiero.

Debido a la presencia de valores extremos, se muestran dos lecturas complementarias: una en escala original, para reconocer la magnitud de la cola, y otra acotada al percentil 99, para observar con mayor claridad el comportamiento habitual.

In [ ]:
# Histograma en escala original.
# Este grafico muestra la existencia de una cola larga, aunque no sea el mas comodo para leer el centro de la distribucion.
plt.figure(figsize=(9, 4))
ax = sns.histplot(data=datos_grafico, x="amount", bins=50, color="#4C78A8")
ax.set_title("Distribucion del monto de las transacciones - escala original")
ax.set_xlabel("Monto")
ax.set_ylabel("Frecuencia")
plt.show()

**Interpretacion:** el analisis de la distribucion de `amount` confirma la presencia de una alta asimetria positiva.

La mayor parte de las transacciones se concentra en montos bajos, mientras que un grupo reducido de operaciones alcanza valores considerablemente mayores. Estos extremos son relevantes, pero dificultan la visualizacion general de la distribucion.

In [ ]:
# Histograma acotado al percentil 99 para observar mejor el comportamiento habitual.
# No se eliminan datos del analisis: el filtro se usa solo para visualizacion.
p99_amount = datos["amount"].quantile(0.99)
datos_monto_p99 = datos_grafico[datos_grafico["amount"] <= p99_amount]

plt.figure(figsize=(9, 4))
ax = sns.histplot(data=datos_monto_p99, x="amount", bins=50, color="#4C78A8")
ax.set_title("Distribucion del monto hasta el percentil 99")
ax.set_xlabel("Monto")
ax.set_ylabel("Frecuencia")
plt.show()

**Interpretacion:** al limitar la visualizacion al percentil 99 se aprecia mejor la zona donde se concentra la operacion habitual.

Este ajuste no elimina registros del analisis; solo permite interpretar visualmente la distribucion sin que los valores extremos dominen por completo el grafico.

## 6. Analisis de outliers en montos

Los valores atipicos en variables monetarias deben analizarse con especial cuidado.

En un problema de fraude, un outlier puede representar ruido, una transaccion legitima de alto valor o precisamente un comportamiento que merece investigacion adicional. Por esta razon, en esta etapa no se eliminan registros.

In [ ]:
# Boxplot general para visualizar valores extremos en amount.
plt.figure(figsize=(9, 3))
ax = sns.boxplot(data=datos_grafico, x="amount", color="#72B7B2")
ax.set_title("Valores atipicos en el monto de transaccion - escala original")
ax.set_xlabel("Monto")
plt.show()

In [ ]:
# Deteccion simple de outliers con rango intercuartilico (IQR).
# No se eliminan registros; solo se cuantifica su presencia.
q1 = datos["amount"].quantile(0.25)
q3 = datos["amount"].quantile(0.75)
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

outliers_amount = datos[(datos["amount"] < limite_inferior) | (datos["amount"] > limite_superior)]

pd.DataFrame(
    {
        "metrica": ["limite_inferior", "limite_superior", "cantidad_outliers", "porcentaje_outliers"],
        "valor": [
            limite_inferior,
            limite_superior,
            len(outliers_amount),
            len(outliers_amount) / len(datos) * 100,
        ],
    }
)

In [ ]:
# Cuantiles utiles para dimensionar los extremos sin eliminarlos.
datos["amount"].quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00]).to_frame("amount")

**Interpretacion:** el boxplot de `amount` evidencia una gran cantidad de valores atipicos.

En este contexto, los outliers no deben eliminarse automaticamente. En problemas de fraude, estos valores pueden representar precisamente los eventos de interes o transacciones que requieren mayor revision. Su tratamiento debe realizarse con cautela en etapas posteriores.

## 7. Comparacion entre fraude y no fraude

Una vez identificada la distribucion general de los montos, se compara el comportamiento de `amount` entre transacciones fraudulentas y no fraudulentas.

El objetivo no es asumir que todo fraude ocurre en montos altos, sino evaluar si existen diferencias descriptivas entre ambas clases.

In [ ]:
# Comparacion estadistica simple entre fraude y no fraude.
comparar_fraude_vs_no_fraude(datos, "amount").rename(index=ETIQUETAS_FRAUDE)

In [ ]:
# Boxplot por clase en escala original.
# La cola extrema puede comprimir las cajas, pero permite ver la magnitud total de los valores atipicos.
plt.figure(figsize=(8, 4))
ax = sns.boxplot(
    data=datos_grafico,
    x="clase_fraude",
    hue="clase_fraude",
    y="amount",
    palette=PALETA_FRAUDE,
    legend=False,
)
ax.set_title("Monto segun condicion de fraude - escala original")
ax.set_xlabel("Clase")
ax.set_ylabel("Monto")
plt.show()

**Interpretacion:** en escala original, los valores extremos dominan la visualizacion y comprimen la zona central del grafico.

Esta vista permite dimensionar la magnitud total de los montos, pero no es suficiente para comparar con claridad el comportamiento tipico entre fraude y no fraude.

In [ ]:
# Boxplot acotado al percentil 99 para comparar mejor fraude vs no fraude.
# El filtro es solo visual; los datos originales se mantienen intactos.
datos_comparacion_p99 = datos_grafico[datos_grafico["amount"] <= p99_amount]

plt.figure(figsize=(8, 4))
ax = sns.boxplot(
    data=datos_comparacion_p99,
    x="clase_fraude",
    hue="clase_fraude",
    y="amount",
    palette=PALETA_FRAUDE,
    legend=False,
)
ax.set_title("Monto segun condicion de fraude - hasta percentil 99")
ax.set_xlabel("Clase")
ax.set_ylabel("Monto")
plt.show()

**Interpretacion:** la version acotada al percentil 99 permite comparar con mayor claridad las medianas y rangos frecuentes de ambas clases.

Si las transacciones fraudulentas mantienen montos superiores aun sin considerar los extremos mas altos, `amount` se convierte en una variable relevante para seguir explorando dentro del analisis descriptivo.

**Patron identificado: montos elevados asociados a fraude**

El analisis comparativo muestra que las transacciones fraudulentas tienden a presentar montos significativamente mayores en comparacion con las transacciones normales.

Este comportamiento es coherente con escenarios reales, donde el fraude puede buscar maximizar el beneficio economico en una sola operacion o en un numero reducido de transacciones.

**Lectura de negocio:** las transacciones de alto monto deben considerarse de mayor riesgo, especialmente cuando aparecen combinadas con otros factores sospechosos, como tipo de transaccion, destino o comportamiento inusual del cliente.

## 8. Analisis por tipo de transaccion

La variable `type` permite segmentar las operaciones segun el tipo de transaccion.

Este analisis es especialmente util desde una perspectiva de negocio, ya que permite identificar si el fraude se concentra en operaciones especificas y no de forma uniforme en todo el sistema.

In [ ]:
# Distribucion general por tipo de transaccion.
distribucion_variable(datos, "type")

In [ ]:
# Tasa de fraude por tipo de transaccion.
fraude_por_tipo = tasa_fraude_por_grupo(datos, "type")
fraude_por_tipo

In [ ]:
# Grafico enfocado en la tasa de fraude por tipo.
plt.figure(figsize=(8, 4))
ax = sns.barplot(
    data=fraude_por_tipo.reset_index(),
    x="type",
    y="tasa_fraude",
    color="#E45756",
)
ax.set_title("Tasa de fraude por tipo de transaccion")
ax.set_xlabel("Tipo de transaccion")
ax.set_ylabel("Tasa de fraude (%)")
plt.xticks(rotation=30)
plt.show()

**Interpretacion:** el analisis por tipo de transaccion muestra que no todas las categorias presentan el mismo nivel de exposicion al fraude.

Esto sugiere que ciertos tipos de operaciones concentran mayor riesgo, lo cual es consistente con escenarios reales donde el fraude tiende a ocurrir en operaciones especificas, como transferencias o retiros. Para priorizar controles, conviene mirar tanto el volumen de operaciones como la tasa de fraude.

**Patron identificado: concentracion del fraude en transferencias**

Los casos de fraude se concentran principalmente en operaciones de tipo transferencia.

Este resultado es consistente con contextos reales donde las estafas telefonicas, la ingenieria social o la suplantacion de identidad suelen inducir a las victimas a transferir dinero hacia cuentas fraudulentas.

**Lectura de negocio:** las transferencias representan un canal prioritario de riesgo. Su monitoreo deberia considerar monto, frecuencia, cuenta destino y desviaciones respecto al comportamiento habitual del cliente.

## 9. Analisis temporal

La columna `step` representa el avance temporal de la simulacion. A partir de esta variable se construye una hora simulada para revisar si la tasa de fraude cambia segun el momento del dia.

Este analisis no busca afirmar un patron operativo real, sino explorar si existe concentracion temporal dentro de la simulacion.

In [ ]:
# Variables temporales simples derivadas de step.
datos_eda = crear_variables_temporales(datos)

fraude_por_hora = tasa_fraude_por_grupo(datos_eda, "hora_simulada")
fraude_por_hora.head(10)

In [ ]:
# Evolucion de la tasa de fraude por hora simulada.
plt.figure(figsize=(10, 4))
serie_hora = fraude_por_hora.sort_index().reset_index()
ax = sns.lineplot(data=serie_hora, x="hora_simulada", y="tasa_fraude", marker="o")
ax.set_title("Tasa de fraude por hora simulada")
ax.set_xlabel("Hora simulada")
ax.set_ylabel("Tasa de fraude (%)")
plt.show()

**Interpretacion:** si algunas horas simuladas presentan mayor tasa de fraude, esto podria orientar futuras visualizaciones o reglas de monitoreo temporal.

Dado que PaySim es un dataset sintetico, este resultado debe interpretarse como un patron exploratorio y no como una conclusion directa sobre comportamiento real de clientes.

## 10. Correlacion entre variables

La correlacion permite revisar relaciones lineales entre variables numericas.

En este bloque se analiza primero la relacion de cada variable con `isFraud` y luego se presenta una matriz de correlacion para observar relaciones generales entre montos, saldos y variables derivadas.

In [ ]:
# Correlaciones de variables numericas con la variable objetivo.
correlaciones_fraude = (
    datos_eda.select_dtypes(include=np.number)
    .corr()["isFraud"]
    .drop("isFraud")
    .sort_values(key=abs, ascending=False)
    .to_frame("correlacion_con_fraude")
)

correlaciones_fraude

In [ ]:
# Grafico de correlaciones con la variable objetivo.
plt.figure(figsize=(8, 4))
correlaciones_plot = correlaciones_fraude.reset_index().rename(columns={"index": "variable"})
ax = sns.barplot(
    data=correlaciones_plot,
    x="correlacion_con_fraude",
    y="variable",
    color="#4C78A8",
)
ax.set_title("Correlacion de variables numericas con fraude")
ax.set_xlabel("Correlacion con isFraud")
ax.set_ylabel("Variable")
plt.show()

In [ ]:
# Matriz de correlacion.
# Se usa una muestra si el dataset es grande para mantener el grafico liviano.
variables_numericas = datos_grafico.select_dtypes(include=np.number)
matriz_correlacion = variables_numericas.corr()

plt.figure(figsize=(10, 7))
ax = sns.heatmap(
    matriz_correlacion,
    cmap="RdBu_r",
    center=0,
    annot=True,
    fmt=".2f",
    linewidths=0.5,
)
ax.set_title("Matriz de correlacion de variables numericas")
plt.show()

**Interpretacion:** la matriz de correlacion muestra relaciones fuertes entre variables asociadas a balances, lo cual es esperado por su naturaleza contable.

Sin embargo, no se observan correlaciones lineales fuertes con la variable objetivo `isFraud`. Esto sugiere que el fraude no depende de una sola variable de forma directa, sino de combinaciones de comportamiento, segmentos especificos y posibles valores extremos.

**Patron identificado: baja correlacion individual con fraude**

Ninguna variable presenta una relacion lineal fuerte con el indicador de fraude. Esto indica que el fraude no se explica adecuadamente desde una unica variable aislada.

**Lectura de negocio:** el fraude debe entenderse como un fenomeno multivariable y contextual. Una sola metrica rara vez sera suficiente para identificar todos los casos relevantes.

**Patron identificado: relacion entre montos y balances**

Se observa relacion entre el monto de la transaccion y variables asociadas a balances, especialmente saldos de origen y destino. Esta relacion aporta coherencia operativa al dataset, pero no explica directamente el fraude.

**Lectura de negocio:** los balances ayudan a contextualizar una transaccion, pero deben analizarse junto con otros elementos. Una operacion puede ser contablemente coherente y aun asi formar parte de un comportamiento fraudulento.

**Patron identificado: limitaciones en la deteccion automatica**

La variable `isFlaggedFraud` presenta cierta relacion con el fraude, pero no captura completamente todos los casos identificados por `isFraud`.

**Lectura de negocio:** las reglas automaticas pueden detectar parte del riesgo, pero no todos los escenarios posibles. Esto deja espacio para mejorar reglas de monitoreo y avanzar hacia una etapa posterior de deteccion de anomalias.

## 11. Comentarios finales del EDA

El analisis exploratorio muestra que el fraude en PaySim se comporta como un fenomeno poco frecuente, concentrado y altamente dependiente del contexto transaccional.

Los principales indicios aparecen en transacciones de mayor monto, especialmente en operaciones de transferencia. Al mismo tiempo, las correlaciones individuales con la variable objetivo son limitadas, lo que refuerza la idea de que el fraude no puede explicarse adecuadamente mediante una sola variable.

Desde una perspectiva de negocio, el EDA permite priorizar focos de monitoreo: transferencias, montos elevados, coherencia entre saldos y casos que no son capturados por reglas automaticas.

Estos hallazgos preparan el terreno para una fase posterior de deteccion de anomalias, donde el objetivo sera identificar transacciones que se alejen del comportamiento esperado sin asumir todavia la existencia de un modelo predictivo.